# Baltimore City, MD parcels

Build a Baltimore City-only export from Baltimore City's Open GIS Real Property dataset
(ArcGIS FeatureServer: `CityView/Realproperty_OB`, layer 0).

Data source: https://geodata.baltimorecity.gov/egis/rest/services/CityView/Realproperty_OB/FeatureServer/0

Key fields:
- `BLOCKLOT` — primary parcel ID (block + lot, e.g. `"0001 001"`)
- `CURRLAND` — current assessed land value
- `CURRIMPR` — current assessed improvement/building value
- `USEGROUP` — land use group code (R, C, I, E, M, U, RC, CR, CC, EC)
- `SDATCODE` — 5-digit SDAT numeric land use code
- `VACIND` — vacant indicator ('Y' = vacant)
- `SDATLINK` — pre-built URL to the SDAT record (use this for `link`)
- `LANDEXMP`, `IMPREXMP` — exempt land/improvement values

No jurisdictional filtering needed — dataset is Baltimore City only.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from shapely.ops import unary_union

import sys
sys.path.append("..")
from parcel_calculations import add_improvement_ratio_fields
from cloud_utils import get_feature_data_with_geometry, ensure_geodataframe

# Config
SCRAPE_DATA = 0  # set to 1 to pull a fresh scrape from ArcGIS
DATA_DIR = "data/baltimore"
os.makedirs(DATA_DIR, exist_ok=True)

# ArcGIS FeatureServer for Baltimore City Real Property
# Full endpoint: https://geodata.baltimorecity.gov/egis/rest/services/CityView/Realproperty_OB/FeatureServer/0
base_url = "https://geodata.baltimorecity.gov/egis/rest/services"
dataset_name = "CityView/Realproperty_OB"
layer_id = 0

In [ ]:
if SCRAPE_DATA == 1:
    print("Downloading Baltimore City parcels from ArcGIS FeatureServer (paginated)...")
    parcel_gdf = get_feature_data_with_geometry(
        dataset_name, base_url, layer_id, paginate=True, out_epsg=4326
    )
    if parcel_gdf is None or len(parcel_gdf) == 0:
        raise RuntimeError("No parcels downloaded from Baltimore ArcGIS endpoint.")

    today_str = datetime.now().strftime("%Y_%m_%d")
    out_path = os.path.join(DATA_DIR, f"baltimore_parcels_{today_str}.parquet")
    parcel_gdf.to_parquet(out_path, index=False)
    print(f"✅ Saved new scrape to {out_path}")

else:
    files = glob.glob(os.path.join(DATA_DIR, "baltimore_parcels_*.parquet"))
    if not files:
        raise FileNotFoundError(
            f"No parcel files found in {DATA_DIR}. Set SCRAPE_DATA=1 to scrape."
        )
    files_sorted = sorted(
        files,
        key=lambda x: datetime.strptime(
            os.path.basename(x).replace("baltimore_parcels_", "").replace(".parquet", ""),
            "%Y_%m_%d",
        ),
        reverse=True,
    )
    latest_file = files_sorted[0]
    print(f"✅ Loading most recent scrape: {latest_file}")
    parcel_gdf = pd.read_parquet(latest_file)

parcel_gdf = ensure_geodataframe(parcel_gdf)
print(f"✅ Loaded as {type(parcel_gdf).__name__} | CRS={parcel_gdf.crs} | rows={len(parcel_gdf):,}")

In [ ]:
pd.set_option('display.max_columns', None)
display(parcel_gdf.head())

In [ ]:
# Examine USEGROUP distribution
print("USEGROUP value counts:")
print(parcel_gdf["USEGROUP"].value_counts(dropna=False))
print()
print("VACIND value counts:")
print(parcel_gdf["VACIND"].value_counts(dropna=False))

In [ ]:
# Print top SDATCODE values for context
with pd.option_context('display.max_rows', 60):
    print("Top SDATCODE values:")
    print(parcel_gdf["SDATCODE"].value_counts(dropna=False).head(60))

In [ ]:
# Check for duplicate BLOCKLOTs (condo cure)
n_dupes = parcel_gdf.duplicated(subset=["BLOCKLOT"], keep=False).sum()
print(f"Duplicate rows by BLOCKLOT: {n_dupes}")

if n_dupes > 0:
    # Collapse duplicates: sum numeric value fields, union geometries, take first for categoricals
    numeric_sum_candidates = [
        "CURRLAND", "CURRIMPR", "BFCVLAND", "BFCVIMPR", "TAXBASE", "FULLCASH",
        "LANDEXMP", "IMPREXMP", "Shape__Area", "Shape__Length",
    ]
    numeric_sum_cols = [
        c for c in numeric_sum_candidates if c in parcel_gdf.columns
        and np.issubdtype(parcel_gdf[c].dtype, np.number)
    ]
    categorical_cols = [
        c for c in parcel_gdf.columns
        if c not in set(numeric_sum_cols + ["geometry", "BLOCKLOT"])
    ]

    agg_dict = {c: "sum" for c in numeric_sum_cols}
    agg_dict.update({c: "first" for c in categorical_cols})

    collapsed = (
        parcel_gdf.groupby("BLOCKLOT", dropna=False).agg(agg_dict).reset_index()
    )
    geom_union = parcel_gdf.groupby("BLOCKLOT", dropna=False)["geometry"].apply(
        lambda geoms: unary_union([g for g in geoms if g is not None])
        if any(g is not None for g in geoms) else None
    )
    collapsed["geometry"] = geom_union.values
    parcel_gdf = gpd.GeoDataFrame(collapsed, geometry="geometry", crs=parcel_gdf.crs)
    print(f"✅ Rows after condo-cure collapse: {len(parcel_gdf):,}")
else:
    print("✅ No duplicates — skipping collapse step")

In [ ]:
def categorize_property_type(row):
    """
    Assigns property category based on Baltimore City's USEGROUP and SDATCODE fields.
    USEGROUP codes: R=Residential, C=Commercial, CC=Commercial Condo,
                    CR/RC=Mixed, I=Industrial, E/EC=Exempt, M=Marsh, U=Utility
    Note: USEGROUP values may have trailing spaces — always strip before comparison.
    SDATCODE is 5-digit for Baltimore City (e.g. 11130=Rowhouse, 11110=Single Family).
    """
    usegroup = str(row.get("USEGROUP", "") or "").strip().upper()
    vacind = str(row.get("VACIND", "") or "").strip().upper()
    sdatcode = str(row.get("SDATCODE", "") or "").strip()

    # Vacant indicator overrides everything
    if vacind == "Y":
        return "Vacant Land"

    # Exempt / governmental
    if usegroup in ("E", "EC"):
        return "Exempt / Governmental"

    # Residential — refine by 5-digit SDATCODE
    if usegroup == "R":
        if sdatcode in ("11110", "11115"):                              return "Single Family"
        if sdatcode in ("11120", "11125"):                              return "Semi-Detached"
        if sdatcode in ("11130", "11135", "11136"):                    return "Rowhouse"
        if sdatcode in ("91010", "91020", "91030"):                    return "Condo/PUD"
        if sdatcode in ("11210",):                                     return "Two Family"
        if sdatcode in ("11220",):                                     return "Three Family"
        if sdatcode in ("11230", "11235", "11236", "11310", "11320"): return "Large Multi-Family (4+ units)"
        if sdatcode in ("11140", "11150", "11160"):                    return "Residential Vacant"
        return "Other Residential"

    # Mixed Use residential-leaning
    if usegroup in ("RC", "CR"):
        return "Mixed Use"

    # Commercial — some SDATCODE 46xxx codes are large multi-family apartments
    if usegroup in ("C", "CC"):
        if sdatcode in ("46000", "46100", "46200"):  return "Large Multi-Family (4+ units)"
        if sdatcode in ("44000", "44100"):           return "Parking Garage"
        return "Commercial"

    if usegroup == "I":  return "Industrial"
    if usegroup == "U":  return "Utility"
    if usegroup == "M":  return "Open Space / Natural"

    return "Other"


parcel_gdf["PROPERTY_CATEGORY"] = parcel_gdf.apply(categorize_property_type, axis=1)

with pd.option_context('display.max_rows', None):
    print("PROPERTY_CATEGORY value counts:")
    print(parcel_gdf["PROPERTY_CATEGORY"].value_counts(dropna=False))


In [ ]:
# Filter out fully exempt parcels
# A parcel is 'fully exempt' if it's in the Exempt/Governmental category
# or if its exempt values cover all of its assessed value.
export_gdf = parcel_gdf.copy()

currland = pd.to_numeric(export_gdf.get("CURRLAND"), errors="coerce").fillna(0)
currimpr = pd.to_numeric(export_gdf.get("CURRIMPR"), errors="coerce").fillna(0)
landexmp = pd.to_numeric(export_gdf.get("LANDEXMP"), errors="coerce").fillna(0)
imprexmp = pd.to_numeric(export_gdf.get("IMPREXMP"), errors="coerce").fillna(0)

total_value = currland + currimpr
total_exempt = landexmp + imprexmp

# Flag as exempt if category is Exempt, OR exempt values >= 99.5% of total
exempt_by_category = export_gdf["PROPERTY_CATEGORY"] == "Exempt / Governmental"
exempt_by_value = (total_value > 0) & (total_exempt / total_value >= 0.995)
export_gdf["exemption_flag"] = (exempt_by_category | exempt_by_value).astype(int)

before = len(export_gdf)
export_gdf = export_gdf[export_gdf["exemption_flag"] == 0].copy()
print(f"✅ Removed {before - len(export_gdf):,} fully exempt parcels")
print(f"✅ Rows remaining: {len(export_gdf):,}")

In [ ]:
# =============================
# Standard export block
# =============================

# 1) Land and improvement values
export_gdf["land_value"] = pd.to_numeric(export_gdf.get("CURRLAND"), errors="coerce")
export_gdf["improvement_value"] = pd.to_numeric(export_gdf.get("CURRIMPR"), errors="coerce")
export_gdf["full_market_value"] = export_gdf["land_value"].fillna(0) + export_gdf["improvement_value"].fillna(0)

# 2) Land use fields
export_gdf["property_land_use_category"] = export_gdf["PROPERTY_CATEGORY"]

def categorize_property_refined(row):
    cat = str(row["PROPERTY_CATEGORY"])
    if "Vacant" in cat:
        return "Vacant"
    elif "Parking Garage" in cat:
        return "Parking Lot"
    elif row["improvement_value"] < 0.5 * (row["land_value"] + row["improvement_value"]):
        return "Underdeveloped"
    else:
        return None

export_gdf["property_land_use_refined"] = export_gdf.apply(categorize_property_refined, axis=1)

# 3) Area (compute geodesically from geometry)
from pyproj import Geod

geod = Geod(ellps="WGS84")

def geodesic_area_sqft(geom):
    if geom is None or geom.is_empty:
        return np.nan
    gtype = geom.geom_type
    if gtype == "Polygon":
        lon, lat = geom.exterior.coords.xy
        area_m2, _ = geod.polygon_area_perimeter(lon, lat)
        return abs(area_m2) * 10.763910416709722
    if gtype == "MultiPolygon":
        return sum(geodesic_area_sqft(p) for p in geom.geoms)
    return np.nan

export_gdf["geometry"] = export_gdf["geometry"].apply(
    lambda g: g if g is None or g.is_valid else g.buffer(0)
)
export_gdf["area_sqft"] = export_gdf["geometry"].apply(geodesic_area_sqft)
export_gdf.loc[export_gdf["area_sqft"] < 1, "area_sqft"] = np.nan

# 4) Per-sqft metrics
export_gdf["full_market_value_per_sqft"] = export_gdf["full_market_value"] / export_gdf["area_sqft"]
export_gdf["land_value_per_sqft"] = export_gdf["land_value"] / export_gdf["area_sqft"]
export_gdf["improvement_value_per_sqft"] = export_gdf["improvement_value"] / export_gdf["area_sqft"]

# 5) Improvement/land ratio fields
export_gdf = add_improvement_ratio_fields(
    export_gdf,
    land_col="land_value",
    improvement_col="improvement_value",
)

# 6) Parcel link — use pre-built SDATLINK field from Baltimore City data
if "SDATLINK" in export_gdf.columns:
    export_gdf["link"] = export_gdf["SDATLINK"].astype(str).str.strip()
else:
    # Fallback: construct from WARD, SECTION, BLOCK, LOT
    def build_sdat_link(row):
        try:
            ward = int(row.get("WARD", 0))
            # SECTION in the data has a trailing zero (e.g. 370 -> 37)
            section_raw = str(row.get("SECTION", "")).strip().rstrip("0")
            section = section_raw if section_raw else "0"
            block = str(row.get("BLOCK", "")).strip()
            lot = str(row.get("LOT", "")).strip()
            return (
                f"http://sdat.dat.maryland.gov/realproperty/pages/viewdetails.aspx"
                f"?County=03&SearchType=ACCT&Ward={ward}&SECTION={section}"
                f"&BLOCK={block}&LOT={lot}"
            )
        except Exception:
            return np.nan
    export_gdf["link"] = export_gdf.apply(build_sdat_link, axis=1)

print("✅ Export block complete")
print("Refined category counts:")
print(export_gdf["property_land_use_refined"].value_counts(dropna=False))

In [ ]:
# Select and rename columns for canonical export schema
columns_to_export = [
    "geometry",
    "exemption_flag",
    "property_land_use_category",
    "property_land_use_refined",
    "full_market_value",
    "full_market_value_per_sqft",
    "land_value",
    "land_value_per_sqft",
    "improvement_value",
    "improvement_value_per_sqft",
    "TLLDIMPROV",
    "IMPR_LAND_RATIO",
    "IMPR_LAND_PCT",
    "IMPR_PCT_TOTAL",
    "link",
]

for col in columns_to_export:
    if col not in export_gdf.columns:
        export_gdf[col] = np.nan

export_final = export_gdf[columns_to_export].rename(columns={
    "land_value": "current_full_land_value"
})

# Ensure geometry validity
export_final["geometry"] = export_final["geometry"].apply(
    lambda geom: geom if geom is None or geom.is_valid else geom.buffer(0)
)

# Ensure CRS is EPSG:4326
export_final = gpd.GeoDataFrame(export_final, geometry="geometry", crs=export_gdf.crs)
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.to_crs("EPSG:4326")
    print("✅ Converted to EPSG:4326")

# Save canonical + dated parquets
canonical_path = os.path.join(DATA_DIR, "baltimore-md-parcels.parquet")
today_str = datetime.now().strftime("%Y_%m_%d")
dated_path = os.path.join(DATA_DIR, f"baltimore-md-parcels_{today_str}.parquet")

export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)

print(f"✅ Saved export parquet: {canonical_path}")
print(f"✅ Also saved dated version: {dated_path}")
print("Export columns:", export_final.columns.tolist())
print(f"\nTotal rows exported: {len(export_final):,}")

In [ ]:
# Upload to dev Azure blob
upload_dev = True

if upload_dev:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING or update connection_string before upload."
        )

    container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    blob_name = "baltimore-md-parcels.parquet"
    local_path = os.path.join(DATA_DIR, blob_name)

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local parquet not found: {local_path}")

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service.get_container_client(container)

    with open(local_path, "rb") as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)

    print(f"✅ Uploaded {local_path} -> {container}/{blob_name}")
else:
    print("upload_dev is False; skipping upload.")

In [ ]:
# Promote dev blob to prod
promote_to_prod = False  # set True once data is validated
promote_overwrite = True

if promote_to_prod:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING or update connection_string before promotion."
        )

    dev_container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    prod_container = os.getenv("AZURE_PROD_CONTAINER", "parquets-prod")
    blob_name = "baltimore-md-parcels.parquet"

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    dev_blob = blob_service.get_blob_client(dev_container, blob_name)
    prod_blob = blob_service.get_blob_client(prod_container, blob_name)

    if not dev_blob.exists():
        raise FileNotFoundError(f"Dev blob not found: {dev_container}/{blob_name}")

    if prod_blob.exists():
        if not promote_overwrite:
            raise FileExistsError(
                "Prod blob already exists. Set promote_overwrite=True to replace it."
            )
        prod_blob.delete_blob()

    prod_blob.start_copy_from_url(dev_blob.url)
    print(f"✅ Promoted {dev_container}/{blob_name} -> {prod_container}/{blob_name}")
else:
    print("promote_to_prod is False; skipping promotion.")